# Pixel sensitivity of depth reconstruction

Paper reference: §3.2 (System Model), §3.3.6 (Post-processing).

For each target distance along the reference laser ray, how far (in
image pixels) can the laser-dot label be from the true pixel location
before the reconstructed depth exceeds a given percent error? We
brute-force the question: at each distance $z$, reconstruct from every
pixel in the image, measure percent depth error, and find the
smallest radius around the true label that first exceeds 1 / 5 / 10 /
15 / 20 % error.

This is a labeling-tolerance bound. The laser-label noise measured in
`notebooks/laser_labeling/laser_label_analysis.ipynb` on real field
data is ~(3, 5) px; the curves here translate that into depth error at
each operating distance.

No paper figure; diagnostic only.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

from fishsense_imwut.camera import reconstruct_points
from fishsense_imwut.constants import FOCAL_LENGTH_PX, IMAGE_HEIGHT, IMAGE_WIDTH

## Setup

In [ ]:
camera_intrinsics = np.array([
    [FOCAL_LENGTH_PX, 0, IMAGE_WIDTH / 2],
    [0, FOCAL_LENGTH_PX, IMAGE_HEIGHT / 2],
    [0, 0, 1],
])
inverted_camera_intrinsics = np.linalg.inv(camera_intrinsics)

laser_position = np.array([-0.04, -0.11, 0])
laser_direction = np.array([1e-10, 1e-10, 1])

In [ ]:
STEP_COUNT = 1000
t = np.linspace(0.5, 30, STEP_COUNT)

p = laser_position[:, np.newaxis] + t[np.newaxis, :] * laser_direction[:, np.newaxis]
s = camera_intrinsics @ (p / p[2, :])
s_pixel = np.round(s)

`s` is the noise-free laser-dot pixel trajectory; `s_pixel` is its integer-pixel version — what a labeler would actually report. The sweep runs from 0.5 m (closest usable depth) to 30 m.

## Brute-force error radius

At each sampled depth $z$, reconstruct from every pixel in the image
and record the minimum pixel distance from `s_pixel` at which percent
depth error first exceeds each threshold. The sweep is capped at $z =
5$ m because the useful operating range of the device lives below
that; beyond 5 m the denominator of Equation 5 is already too small
to trust (see `../max_distance.ipynb`).

In [ ]:
x_range = np.arange(0, IMAGE_WIDTH)
y_range = np.arange(0, IMAGE_HEIGHT)
xy_grid = np.array(np.meshgrid(x_range, y_range, indexing='ij'))
xy_homogeneous = np.vstack((
    xy_grid.reshape(2, -1),
    np.ones((1, xy_grid.shape[1] * xy_grid.shape[2])),
))

thresholds = [1, 5, 10, 15, 20]
error_distances = {k: [] for k in thresholds}
z_values = []

for i in tqdm(range(s_pixel.shape[1])):
    z = p[2, i]
    if z < 0.5:
        continue

    p_reconstructed, _ = reconstruct_points(
        xy_homogeneous, inverted_camera_intrinsics, laser_position, laser_direction,
    )
    percent_errors = np.abs(p_reconstructed[2, :] - z) / z * 100
    distances = np.linalg.norm(
        xy_homogeneous[:2, :] - s_pixel[:2, i:i + 1], axis=0,
    )

    for k in thresholds:
        above = distances[percent_errors > k]
        error_distances[k].append(above.min() if above.size else np.nan)

    z_values.append(z)
    if z > 5:
        break

z_values = np.array(z_values)
for k in thresholds:
    error_distances[k] = np.array(error_distances[k])

In [ ]:
fig, ax = plt.subplots()
for k in thresholds:
    ax.plot(z_values, error_distances[k], label=f'{k}% error')
ax.set_xlabel('Target distance $z$ (m)')
ax.set_ylabel('Pixel distance to error threshold')
ax.set_title('Pixel-label tolerance vs. target distance')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)
fig

Each curve is the largest pixel offset a labeler can make (relative to the true laser-dot pixel) before the reconstructed depth crosses that percent-error line. Curves drop steeply with distance: parallax shrinks, so a one-pixel shift maps to an increasingly large depth change. The reference labeling noise of ~(3, 5) px starts to clear the 10 % line around 3–4 m.